# Entity Linking and Information Retrieval Debugging with DeepFix

This tutorial demonstrates how to use DeepFix to diagnose and debug **Information Retrieval (IR)** and **Entity Linking** models. In these tasks:
- **Queries** (e.g., search queries or text mentions) are matched against a large corpus of **Documents** or **Entities**.
- The goal is to retrieve the most relevant documents/entities for each query.
- Evaluating these systems involves analyzing query-document pairs, relevance labels, and retrieval ranks/scores.

DeepFix provides native support for IR and Entity Linking datasets via the `InformationRetrievalDataset` class, allowing you to ingest your data, wrap your model with `IRLookupModel`, and get automated, detailed diagnostic reports (covering data drift, annotation quality, and retrieval performance).

In [20]:
import os
import random
import pandas as pd
import numpy as np
import pyterrier as pt
import hashlib

from deepfix_sdk import DeepFixClient
from deepfix_sdk.data.datasets import InformationRetrievalDataset
from deepfix_sdk.models import IRLookupModel

In [ ]:
# Set up your DeepFix API key.
# Get your API key by signing up at https://deepfix.delcaux.com
os.environ["DEEPFIX_API_KEY"] = ""

In [22]:
# Initialize the DeepFix client
client = DeepFixClient(timeout=300)

## Loading and Preparing the Dataset

We will load a subset of the **BEIR scidocs** dataset using `pyterrier`.
To evaluate the retrieval model, we will:
1. Load queries (topics), ground truth relevance labels (qrels), and documents (corpus).
2. Generate deterministic random embeddings for queries and documents.
3. Simulate retrieval scores for a model to evaluate.

In [23]:
def simulate_retrievals(qrels_df: pd.DataFrame, retrieval_rate: float = 0.8, seed: int = 42) -> pd.DataFrame:
    """Simulate model retrievals from a qrels DataFrame.
    
    Only a fraction of the actual relevant documents/entities are successfully retrieved.
    For retrieved items, we generate simulated confidence scores/probabilities for binary relevance classes.
    """
    rng = np.random.default_rng(seed)
    n = len(qrels_df)

    # Only a fraction of items are "found" by the model
    retrieved_mask = rng.random(n) < retrieval_rate
    df = qrels_df[retrieved_mask].copy()
    m = len(df)

    prob = rng.random(m)
    is_relevant = df["relevance"].values == 1
    # 80% of relevant docs get a high class-1 score, 20% get a low one
    high_class1 = rng.random(m) > 0.2

    score_class0 = np.where(
        is_relevant,
        np.where(high_class1, prob * 0.5, 1 - prob * 0.5),
        1 - prob * 0.2,
    )
    score_class1 = np.where(
        is_relevant,
        np.where(high_class1, 1 - prob * 0.5, prob * 0.5),
        prob * 0.2,
    )

    return pd.DataFrame(
        {
            "query_id": df["query_id"].values,
            "doc_id": df["doc_id"].values,
            "score": [[s0, s1] for s0, s1 in zip(score_class0, score_class1)],
            "relevance": (score_class1 > score_class0).astype(int),
            "rank": rng.integers(1, 101, size=m),
        }
    )


In [24]:
def random_embedder(text: str, dim: int = 10) -> np.ndarray:
    """Compute a deterministic random embedding for a given text.
    
    Useful for demonstrating embedding-based validation (like embedding drift)
    without needing an external LLM or heavy deep learning library.
    """
    seed = int(hashlib.md5(text.encode()).hexdigest(), 16) % (2**32)
    rng = np.random.default_rng(seed)
    return rng.random(dim)

In [25]:
def load_ir_data(subset_queries: int = 50):
    """Load BEIR dbpedia-entity data using PyTerrier and prepare IR datasets."""
    name = "irds:beir/scidocs"
    dataset = pt.get_dataset(name)

    # 1. Get all topics and qrels, subset for fast execution
    all_topics = dataset.get_topics()
    all_qrels = dataset.get_qrels()

    qid_subset = all_topics["qid"].unique()[:subset_queries]
    topics_df = all_topics[all_topics["qid"].isin(qid_subset)]
    qrels_df = all_qrels[all_qrels["qid"].isin(qid_subset)]

    # 2. Build a single dataset, then split using stratified sampling on labels
    ir_ds = InformationRetrievalDataset(
        dataset_name=name,
        topics=topics_df,
        qrels=qrels_df,
        corpus_iter=dataset.get_corpus_iter,
    )

    train_ir_ds, test_ir_ds = ir_ds.split(train_size=0.7, random_state=42)

    # 3. Simulate retrievals and set predictions
    train_ir_ds.set_predictions(simulate_retrievals(train_ir_ds.qrels))
    test_ir_ds.set_predictions(simulate_retrievals(test_ir_ds.qrels))

    # 4. Set embeddings for diagnostic suites
    train_ir_ds.set_embeddings(random_embedder)
    test_ir_ds.set_embeddings(random_embedder)

    return train_ir_ds, test_ir_ds

In [26]:
# Load the dataset
# Subset to 10 queries for a super fast and light demonstration
train_data, test_data = load_ir_data(subset_queries=10)
print(f"Data loaded! Train pairs: {len(train_data)}, Test pairs: {len(test_data)}")

There are multiple query fields available: ('text', 'authors', 'year', 'cited_by', 'references'). To use with pyterrier, provide variant or modify dataframe to add query column.


beir/scidocs documents:   0%|          | 0/25657 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:00<00:00, 13.71it/s]


beir/scidocs documents:   0%|          | 0/25657 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 13.80it/s]


Data loaded! Train pairs: 208, Test pairs: 90


## Defining the IRLookupModel

In Information Retrieval and Entity Linking workflows, predictions (relevance scores, retrieval ranks) are usually pre-calculated by the retrieval engine (e.g., PyTerrier, BM25, or a bi-encoder).

DeepFix provides an `IRLookupModel` class that satisfies scikit-learn's estimator interface by looking up these pre-computed relevance predictions and probabilities for query-document pairs on the fly. This allows you to evaluate your pre-computed retrievals directly through our diagnostic pipeline.

In [27]:
# Initialize lookup model
model = IRLookupModel(train_dataset=train_data, test_dataset=test_data)
model_name = "simulated_retrieval_model"

## Running DeepFix Diagnostic Analysis

Now we will run the automated diagnosis. The `DeepFixClient` will:
1. Ingest the datasets and the lookup model.
2. Trigger the Deepchecks diagnostic suites on the datasets to analyze metadata, data distribution, and text embeddings.
3. Call the model evaluator to measure precision, recall, and potential leakage or overfitting.
4. Synthesize all findings and provide a prioritized list of recommendations.

In [30]:
# Run automated diagnosis
result = client.get_diagnosis(
    train_data=train_data,
    test_data=test_data,
    model=model,
    model_name=model_name,
    language="english"
)

== == == == == == == == == ==  NLP == == == == == == == == == == 


deepchecks - WARNING - Could not find model's classes, using the observed classes. In order to make sure the classes used by the model are inferred correctly, please use the model_classes argument




== == == == == == == == == ==  tabular == == == == == == == == == == 


Submitting analysis job to: https://deepfix.delcaux.com/api/v2/analyse

Request accepted (ID: job_20260518145605_7e3d1632)

Output()

v Analysis complete!

## Reviewing Diagnostic Results

Once analysis is complete, we can visualize the prioritized findings and recommendations in plain text using the `.to_text()` method.

In [31]:
# Display summary of findings and recommended actions
result.to_text()

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The integrated cross-artifact analysis reveals a fundamentally non-functional system with critical blockers that     │
│ must be resolved before any meaningful model training or evaluation. The primary issue is a task type mismatch: the  │
│ dataset is structured for information retrieval/ranking (query-document pairs, relevance labels, retrieval-specific  │
│ features) but labeled as text classification, invalidating the entire modeling approach. This is compounded by       │
│ severe label leakage through retrieval-derived features (l1_norm, cosine_sim) that directly encode relevance         │
│ signals, and a non-functional model checkpoint (IRLookupModel) that has no weights and no provided data source. The  │
│ dataset is critically small (208 samples, 10 queries) with severe class imbalance (~16% positive), and Deepchecks    │
│ confirms significant distribution shifts between train/test (52% new categories in doc_token_count, 0.39 PPS change  │
│ in Max Word Length-label correlation), high feature multicollinearity (7 pairs >0.9), and data quality issues        │
│ (outliers, missing values, frequent substrings). The priority actions are: (1) re-label the task as IR/ranking with  │
│ appropriate metrics (NDCG, MAP), (2) remove leaked features (l1_norm, cosine_sim), (3) provide the missing data      │
│ source and configuration for the IRLookupModel, (4) augment the dataset to a meaningful size, and (5) run complete   │
│ model evaluation checks. Until these critical blockers are resolved, any analysis or training using these artifacts  │
│ will produce invalid results.                                                                                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  16                                                           
 Severity Distribution           HIGH: 8  MEDIUM: 5  LOW: 3                                   

                                  HIGH Severity Issues (8)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical task type mismatch: dataset     │ Re-label the task as information         │
│     │ structured for information               │ retrieval/ranking. Use ranking-specific  │
│     │ retrieval/ranking but labeled as text    │ evaluation metrics (NDCG@k, MAP, MRR)    │
│     │ classification                           │ and loss functions (pairwise ranking     │
│     │ Evidence: Dataset contains queries (10   │ loss, triplet loss). Restructure the     │
│     │ unique), relevance labels, doc_id,       │ pipeline to handle query-document pairs  │
│     │ query_id, and retrieval-specific         │ rather than independent classification   │
│     │ features (l1_norm, cosine_sim). The      │ samples.                                 │
│     │ IRLookupModel is a lookup wrapper        │ Treating query-document pairs as         │
│     │ designed for IR with pre-computed        │ independent classification samples       │
│     │ scores. Cross-referencing: Dataset       │ ignores the fundamental ranking          │
│     │ analyzer identifies the mismatch, Model  │ structure, leading to incorrect          │
│     │ analyzer confirms the model is an IR     │ evaluation, inappropriate loss           │
│     │ lookup, and Deepchecks shows             │ functions, and suboptimal model          │
│     │ distribution shifts consistent with IR   │ training. This is the root cause of      │
│     │ features.                                │ multiple downstream issues.              │
│ 2   │ Severe label leakage through             │ Immediately remove l1_norm and           │
│     │ retrieval-derived features (l1_norm,     │ cosine_sim features from all training,   │
│     │ cosine_sim)                              │ validation, and test data. If these      │
│     │ Evidence: Dataset analyzer: l1_norm and  │ features are needed for evaluation,      │
│     │ cosine_sim are likely computed from a    │ compute them from a separate held-out    │
│     │ simulated retrieval model and could      │ retrieval model that had no access to    │
│     │ encode the relevance signal directly     │ relevance labels.                        │
│     │ (100% unique values). Deepchecks         │ Including leaked features invalidates    │
│     │ analyzer: These features are in the top  │ any model evaluation, as the model can   │
│     │ 7 highly correlated pairs (>0.9) with    │ achieve artificially perfect performance │
│     │ doc_token_count, suggesting they capture │ by learning the relevance signal         │
│     │ length-related information but may also  │ directly from these features rather than │
│     │ encode relevance directly.               │ from actual document content. This is a  │
│     │                                          │ hard blocker for any meaningful          │
│     │                                          │ analysis.                                │
│ 3   │ Model checkpoint (IRLookupModel) is      │ Provide the actual pre-computed scores   │
│     │ non-functional: no weights, no data      │ or relevance data that the IRLookupModel │
│     │ source, no configuration                 │ will reference (e.g., CSV, Parquet,      │
│     │ Evidence: Model analyzer confirms: no    │ database connection). Add a complete     │
│     │ weight files (.bin, .safetensors, .pt)   │ configuration including: lookup table    │
│     │ are provided; the model is a lookup      │ schema, input format (query-doc IDs or   │
│     │ wrapper with no internal parameters. The │ raw text), output format (scores or      │
│     │ hyperparameters only list classes ['0',  │ l

                                 MEDIUM Severity Issues (5)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ High multicollinearity: 7 feature pairs  │ Reduce feature redundancy by removing or │
│     │ with correlation >0.9, all involving     │ combining highly correlated features.    │
│     │ doc_token_count                          │ Keep only doc_token_count as a           │
│     │ Evidence: Deepchecks analyzer: The       │ representative length feature and drop   │
│     │ 'Feature Feature Correlation' check      │ the rest (Text Length, Max Word Length,  │
│     │ identified 7 pairs with correlation      │ etc.). Consider using dimensionality     │
│     │ >0.9, including doc_token_count with     │ reduction (PCA) for the remaining        │
│     │ Text Length, Max Word Length, % Special  │ features.                                │
│     │ Characters, % Punctuation, Lexical       │ High multicollinearity inflates model    │
│     │ Density, Average Words Per Sentence,     │ variance and makes interpretation        │
│     │ cosine_sim, and l1_norm.                 │ unreliable. It also indicates many       │
│     │                                          │ features capture the same underlying     │
│     │                                          │ property (document length), adding       │
│     │                                          │ complexity without benefit.              │
│ 2   │ Text property outliers: 5.56% outliers   │ Inspect the outlier samples to determine │
│     │ in Max Word Length and 6.25% in %        │ their validity. For Max Word Length      │
│     │ Special Characters                       │ outliers, check for unusually long words │
│     │ Evidence: Deepchecks analyzer: Two 'Text │ (e.g., concatenated text, URLs, chemical │
│     │ Property Outliers' checks failed, with   │ formulas). For % Special Characters      │
│     │ outlier ratios exceeding the 5%          │ outliers, check for corrupted text or    │
│     │ threshold. These may indicate data       │ non-standard formatting. Cap, remove, or │
│     │ quality issues such as concatenated      │ correct as appropriate.                  │
│     │ text, URLs, or scraping artifacts.       │ Outliers can distort model training and  │
│     │                                          │ evaluation, especially if they are rare  │
│     │                                          │ but extreme. The presence of outliers in │
│     │                                          │ two different properties suggests data   │
│     │                                          │ quality issues that should be addressed. │
│ 3   │ Missing values in derived text features: │ Investigate the specific documents with  │
│     │ 8 missing in train, 2 in test for        │ missing feature values. Impute with      │
│     │ Sentiment, Subjectivity, Reading Ease    │ median values if the missingness is      │
│     │ Evidence: Dataset analyzer: Sentiment,   │ random, or drop these rows if they are   │
│     │ Subjectivity, Reading Ease each have     │ problematic. Document the reason         │
│     │ count=200 in train (out of 208) and      │ features could not be computed.          │
│     │ count=88 in test (out of 90). These 8    │ Missing values can cause errors during   │
│     │ missing values may be from very short or │ model training or inference.             │
│     │ non-English texts that cannot be         │ Understanding the pattern of missingness │
│     │ processed.                               │ is important to avoid introducing bias.  │
│ 4   │ Frequent substrings detected in text     │ Examine the frequent substrings to       │
│     │ data (4-11 substrings above threshold)   │ d

                                   LOW Severity Issues (3)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Limited vocabulary size relative to      │ Check vocabulary overlap between train   │
│     │ typical scientific text                  │ and test sets. If overlap is very high   │
│     │ Evidence: Dataset analyzer: Train        │ (>90%), the train/test split may not be  │
│     │ vocabulary is 7,963 for 208 documents,   │ truly independent. Consider using a      │
│     │ test vocabulary is 4,876 for 90          │ proper temporal or random split of the   │
│     │ documents. This is relatively small for  │ full SciDocs dataset.                    │
│     │ scientific text, suggesting either short │ High vocabulary overlap is expected in   │
│     │ documents or significant overlap between │ retrieval (same domain), but extremely   │
│     │ train and test.                          │ high overlap may indicate data leakage   │
│     │                                          │ across the split, further invalidating   │
│     │                                          │ evaluation results.                      │
│ 2   │ Class labels lack semantic meaning and   │ Provide a clear label mapping (e.g.,     │
│     │ documentation                            │ 0='not relevant', 1='relevant') and      │
│     │ Evidence: Model analyzer:                │ document the threshold or scoring method │
│     │ Hyperparameters define classes as        │ if the model outputs continuous scores   │
│     │ strings '0' and '1' with no label        │ that are thresholded.                    │
│     │ mapping (id2label, label2id) or          │ Unclear label semantics reduce           │
│     │ description of what each class           │ interpretability and can lead to         │
│     │ represents. The docstring mentions       │ incorrect evaluation. This is a minor    │
│     │ 'relevance' but does not confirm binary  │ usability issue but important for        │
│     │ relevance grading.                       │ reproducibility.                         │
│ 3   │ Unknown tokens present in tokenized data │ Identify the specific unknown tokens and │
│     │ (0.0042% and 0.0019% ratios)             │ decide whether to add them to the        │
│     │ Evidence: Deepchecks analyzer: The       │ tokenizer vocabulary or replace them     │
│     │ 'Unknown Tokens' check failed twice with │ with a special [UNK] token. The very low │
│     │ very low ratios. While minimal, unknown  │ ratio suggests minimal impact.           │
│     │ tokens can cause the model to lose       │ Unknown tokens can cause the model to    │
│     │ information on specific samples.         │ lose information or produce              │
│     │                                          │ unpredictable outputs for a few samples. │
│     │                                          │ While the ratio is tiny, it may affect   │
│     │                                          │ critical test cases disproportionately.  │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''